# Basic RAG from Scratch## Building a Complete RAG SystemLet's build a real RAG system step-by-step!### Components1. Document loader2. Text chunker3. Embedding generator4. Vector store5. Retriever6. Generator (LLM)

In [ ]:
import numpy as npimport pandas as pdfrom typing import List, Dict, Tupleimport jsonimport osfrom pathlib import Pathfrom sentence_transformers import SentenceTransformerimport chromadbfrom chromadb.config import Settings

## Step 1: Load DocumentsIn production, you'd load from files, APIs, databases, etc.

In [ ]:
# Sample knowledge basedocuments = [    {        "id": "doc1",        "content": "Artificial Intelligence (AI) is the simulation of human intelligence by machines. It includes machine learning, natural language processing, and computer vision.",        "metadata": {"source": "ai_basics.txt", "topic": "AI"}    },    {        "id": "doc2",         "content": "Machine Learning is a subset of AI that enables systems to learn from data without explicit programming. Common algorithms include decision trees, neural networks, and support vector machines.",        "metadata": {"source": "ml_guide.txt", "topic": "ML"}    },    {        "id": "doc3",        "content": "RAG (Retrieval-Augmented Generation) combines information retrieval with text generation. It retrieves relevant documents and uses them as context for generating responses.",        "metadata": {"source": "rag_explained.txt", "topic": "RAG"}    },    {        "id": "doc4",        "content": "Vector databases store data as high-dimensional vectors (embeddings). They enable fast similarity search using techniques like HNSW and IVF indexes.",        "metadata": {"source": "vectordb_intro.txt", "topic": "Databases"}    },    {        "id": "doc5",        "content": "Embeddings are numerical representations of text that capture semantic meaning. Similar texts have similar embeddings in vector space.",        "metadata": {"source": "embeddings_101.txt", "topic": "Embeddings"}    }]print(f"Loaded {len(documents)} documents")for doc in documents:    print(f"  - {doc['id']}: {doc['content'][:50]}...")

## Step 2: Create EmbeddingsConvert text to vectors using a pre-trained model

In [ ]:
# Load embedding modelmodel = SentenceTransformer('all-MiniLM-L6-v2')  # 384-dimensional# Generate embeddingstexts = [doc["content"] for doc in documents]embeddings = model.encode(texts)print(f"Embedding shape: {embeddings.shape}")print(f"Dimension: {embeddings.shape[1]}")print(f"\nFirst embedding (truncated):")print(embeddings[0][:10])

## Step 3: Store in Vector DatabaseUsing ChromaDB for local vector storage

In [ ]:
# Initialize ChromaDBclient = chromadb.Client(Settings(    anonymized_telemetry=False,    allow_reset=True))# Create collectioncollection = client.create_collection(    name="knowledge_base",    metadata={"description": "RAG knowledge base"})# Add documentscollection.add(    ids=[doc["id"] for doc in documents],    embeddings=embeddings.tolist(),    documents=texts,    metadatas=[doc["metadata"] for doc in documents])print(f"Added {collection.count()} documents to vector store")

## Step 4: Implement RetrievalQuery the vector database for similar documents

In [ ]:
def retrieve_context(query: str, top_k: int = 3):    """Retrieve relevant documents for a query."""    # Embed query    query_embedding = model.encode([query])[0]        # Search vector DB    results = collection.query(        query_embeddings=[query_embedding.tolist()],        n_results=top_k    )        # Format results    contexts = []    for i in range(len(results['documents'][0])):        contexts.append({            'id': results['ids'][0][i],            'content': results['documents'][0][i],            'distance': results['distances'][0][i],            'metadata': results['metadatas'][0][i]        })        return contexts# Test retrievalquery = "What is machine learning?"results = retrieve_context(query, top_k=2)print(f"Query: {query}\n")for i, result in enumerate(results, 1):    print(f"{i}. Document: {result['id']}")    print(f"   Distance: {result['distance']:.4f}")    print(f"   Content: {result['content'][:100]}...")    print()

## Step 5: Generate ResponseCombine retrieval with LLM generation

In [ ]:
def generate_response(query: str, context_docs: List[Dict]) -> str:    """Generate answer using retrieved context."""    # Build context string    context = "\n\n".join([        f"[Source: {doc['id']}] {doc['content']}"         for doc in context_docs    ])        # Create prompt    prompt = f"""You are a helpful AI assistant. Answer the question based only on the provided context.Context:{context}Question: {query}Answer (cite sources):"""        # In production, send to OpenAI/Anthropic/etc    # For demo, return prompt    return prompt# Generateresponse_prompt = generate_response(query, results)print(response_prompt)print("\n" + "="*70)print("Send this to GPT-4, Claude, or other LLM")

## Complete RAG PipelinePutting it all together

In [ ]:
class SimpleRAG:    """Simple RAG system."""        def __init__(self, embedding_model, vector_store):        self.model = embedding_model        self.collection = vector_store        def query(self, question: str, top_k: int = 3) -> Dict:        """Execute RAG query."""        # 1. Retrieve        contexts = retrieve_context(question, top_k)                # 2. Generate prompt        prompt = generate_response(question, contexts)                # 3. Return (would call LLM here)        return {            'question': question,            'retrieved_docs': contexts,            'prompt': prompt,            'sources': [doc['id'] for doc in contexts]        }# Initialize RAGrag = SimpleRAG(model, collection)# Test queriestest_queries = [    "Explain vector databases",    "What is the difference between AI and ML?",    "How does RAG work?"]for q in test_queries:    print(f"Q: {q}")    result = rag.query(q, top_k=2)    print(f"Sources: {result['sources']}")    print()

## Summary### RAG Pipeline1. **Index Phase** (one-time)   - Load documents   - Generate embeddings   - Store in vector DB2. **Query Phase** (per request)   - Embed user query   - Retrieve similar docs   - Generate response with context### Key Takeaways✅ RAG provides LLMs with relevant context✅ Vector search finds semantically similar docs✅ No fine-tuning needed - works with any LLM✅ Easy to update - just add new documents**Next**: Learn advanced chunking strategies!